# V3.1 full run — 200 pairs, three arms

Runs the whole thing from zero on Colab: downloads its own weights, crops on the GPU,
calls fal for both generation stages, and writes a zip.

| arm | reference | calls |
|---|---|---|
| **BC** | klein makes the wearer bald → CPU crop | 2 |
| **QX** | Qwen regenerates the garment isolated on white | 2 |
| **MQ** | CPU crop → Qwen regenerates it as a **mannequin** — *this is v3.1* | 2 |

**MQ's extraction prompt is assembled per pair** from two CPU reads: the *person's* face
gives a colour word, the *garment crop's* pose gives a framing category, and the category
selects the extent and pose clauses together from one table.

**Why a GPU matters here:** BiRefNet at 1024² is ~49 s per image on a 4-core CPU and
milliseconds on a GPU. 56 crops is the difference between 45 minutes and under a minute.

**Cost.** 200 pairs × 3 arms = 600 klein edits, plus ~56 bald calls and ~256 Qwen
extractions. At the measured klein price of $0.015 that is **~$9.84 of klein**; the Qwen
price is not recorded anywhere in this project and should be checked on the dashboard
before starting. **The run is resumable** — every stage skips what is on disk, so an
exhausted balance costs nothing but a restart.

MQ needs one extraction *per pair* rather than per reference, because the colour word
depends on who is being dressed. That is 200 Qwen calls rather than 56, and it is the
price of the colour reader.


In [ ]:
#@title 1 · Setup — fetch the bundle
# Pulls the 19 MB bundle straight from GitHub. No upload, no full clone.
# If you would rather upload the zip by hand, drop it in /content and this finds that too.
import os, glob, zipfile, urllib.request

URL = ('https://raw.githubusercontent.com/101011101/magichour_takehome/'
       'v3.1-colab-run/v3_colab_bundle.zip')
z = sorted(glob.glob('/content/**/v3_colab_bundle.zip', recursive=True))
if not z:
    print('fetching bundle ...')
    urllib.request.urlretrieve(URL, '/content/v3_colab_bundle.zip')
    z = ['/content/v3_colab_bundle.zip']
print(f'{os.path.getsize(z[0])/1e6:.0f} MB from {z[0]}')

zipfile.ZipFile(z[0]).extractall('/content/v3')
os.chdir('/content/v3')
assert os.path.exists('matrix.csv') and os.path.isdir('testset'), 'bundle looks wrong'
print(f"{len(os.listdir('testset'))-1} test images, matrix present")
print()
print(open('README.md').read()[:700])

In [ ]:
#@title 2 · Dependencies
!pip -q install fal-client mediapipe onnxruntime-gpu opencv-python-headless
import onnxruntime as ort
print('onnxruntime providers:', ort.get_available_providers())

In [ ]:
#@title 3 · Models — reuse a cache if you have one
# Checks, in order: $V3_MODEL_DIR, ./models, /content/models, Drive, ~/.cache/v3_models.
# A file only counts if it is the RIGHT SIZE - an interrupted download passes
# os.path.exists and then fails somewhere much less obvious.
#
# If your copies live somewhere else, set the path here. If you want them to survive a
# runtime restart, mount Drive and set PERSIST - the 224 MB BiRefNet fetch then happens
# once ever rather than once per session.
import os, sys
sys.path.insert(0, 'lib')

V3_MODEL_DIR = ''      #@param {type:"string"}
PERSIST      = ''      #@param {type:"string"}
# e.g. V3_MODEL_DIR = '/content/drive/MyDrive/v3_models'
#      PERSIST      = '/content/drive/MyDrive/v3_models'

if V3_MODEL_DIR: os.environ['V3_MODEL_DIR'] = V3_MODEL_DIR
import v3lib as L
print('searching:'); [print('  ', d) for d in L.search_dirs()]
paths = L.fetch_models(persist=PERSIST or None)
print()
for k, v in paths.items():
    print(f'  {k:9} {os.path.getsize(v)/1e6:7.1f} MB')

In [ ]:
#@title 4 · fal key
import os, getpass
os.environ['FAL_KEY'] = getpass.getpass('FAL_KEY: ')
import fal_client; print('fal client ready')

In [ ]:
#@title 5 · Sanity check — one pair, all three arms, before spending on 200
import sys; sys.path.insert(0, 'lib')
import run_all
run_all.main(matrix='matrix.csv', testset='testset', limit=1)

Check `run/gen/` has three images before continuing. If the crop stage reported
`CPUExecutionProvider`, the GPU is not being used — switch the runtime to GPU and
re-run cell 2, otherwise the 56 crops will take about 45 minutes instead of one.

In [ ]:
#@title 6 · The full run — resumable, safe to re-run after a top-up
import importlib, run_all; importlib.reload(run_all)
have = run_all.main(matrix='matrix.csv', testset='testset')
have

In [ ]:
#@title 7 · Package the results
import shutil, os, json
n = sum(1 for _ in os.scandir('run/gen'))
shutil.make_archive('/content/v31_full_run', 'zip', 'run')
print(f'{n} generated images -> /content/v31_full_run.zip '
      f'({os.path.getsize("/content/v31_full_run.zip")/1e6:.0f} MB)')
from google.colab import files; files.download('/content/v31_full_run.zip')

## What comes back

```
run/inputs/   normalised sources, and {ref}__A4.jpg crops
run/refs/     BC bald frames · QX extractions · MQ mannequins (one per pair)
run/gen/      {set_id}__{BC,QX,MQ}.jpg — the try-ons
run/meta/     every prompt as sent, per-pair colour and framing reads, run.json
```

`meta/mq_prompts.json` records the colour word and framing category chosen for every
pair, so a bad output can be traced to a bad read rather than guessed at.

**Not included: scoring.** The zip is evidence, not a verdict.
